# U.S. Census ACS API Collection and Preparation

This notebook collects 2024 American Community Survey one-year estimates for the living arrangements of adults ages 18–34. It preserves the raw API response, creates readable column names, validates the data, calculates state-level percentages, and exports the cleaned dataset used on the project website.

## 1. Import packages

In [ ]:
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests

## 2. Define the API request

The requested ACS variables describe the total population ages 18–34 and six living-arrangement categories. The API key is entered with `getpass`, so it is not displayed or saved in this notebook.

In [ ]:
base_url = "https://api.census.gov/data/2024/acs/acs1"

variables = [
    "NAME",
    "B09021_008E",  # Total population ages 18-34
    "B09021_009E",  # Lives alone
    "B09021_010E",  # Lives with spouse
    "B09021_011E",  # Lives with unmarried partner
    "B09021_012E",  # Child of householder
    "B09021_013E",  # Lives with other relatives
    "B09021_014E",  # Lives with other nonrelatives
]

api_key = getpass("Paste your Census API key: ").strip()

params = {
    "get": ",".join(variables),
    "for": "state:*",
}

if api_key:
    params["key"] = api_key

print("API endpoint:", base_url)
print("Variables requested:", len(variables))

## 3. Send the GET request

`raise_for_status()` stops the workflow if the Census server returns an unsuccessful HTTP status. The requested URL is not printed because it may contain the API key.

In [ ]:
response = requests.get(base_url, params=params, timeout=30)
response.raise_for_status()

print("Status code:", response.status_code)
print("Content type:", response.headers.get("content-type"))

## 4. Convert the JSON response to a DataFrame and save the raw data

In [ ]:
api_json = response.json()

if not isinstance(api_json, list) or len(api_json) < 2:
    raise ValueError("The Census API did not return the expected table.")

census_raw = pd.DataFrame(api_json[1:], columns=api_json[0])
raw_filename = Path("census_acs_2024_raw.csv")
census_raw.to_csv(raw_filename, index=False)

print("Raw dataset shape:", census_raw.shape)
print("Saved:", raw_filename)
display(census_raw.head())

## 5. Rename columns and convert counts to numeric values

In [ ]:
column_names = {
    "NAME": "state_name",
    "B09021_008E": "total_age_18_34",
    "B09021_009E": "lives_alone",
    "B09021_010E": "lives_with_spouse",
    "B09021_011E": "lives_with_unmarried_partner",
    "B09021_012E": "child_of_householder",
    "B09021_013E": "lives_with_other_relatives",
    "B09021_014E": "lives_with_other_nonrelatives",
    "state": "state_fips",
}

census_clean = census_raw.rename(columns=column_names).copy()
census_clean["state_fips"] = census_clean["state_fips"].astype(str).str.zfill(2)

count_columns = [
    "total_age_18_34",
    "lives_alone",
    "lives_with_spouse",
    "lives_with_unmarried_partner",
    "child_of_householder",
    "lives_with_other_relatives",
    "lives_with_other_nonrelatives",
]

census_clean[count_columns] = census_clean[count_columns].apply(
    pd.to_numeric, errors="raise"
)

## 6. Keep the 50 states and Washington, D.C.

The API returns Puerto Rico as an additional geographic record. It is excluded so the cleaned dataset represents the 50 states and Washington, D.C.

In [ ]:
census_clean = (
    census_clean.loc[census_clean["state_fips"] != "72"]
    .reset_index(drop=True)
)

print("Rows after excluding Puerto Rico:", len(census_clean))

## 7. Validate completeness, uniqueness, and category totals

In [ ]:
category_columns = [
    "lives_alone",
    "lives_with_spouse",
    "lives_with_unmarried_partner",
    "child_of_householder",
    "lives_with_other_relatives",
    "lives_with_other_nonrelatives",
]

category_sum = census_clean[category_columns].sum(axis=1)
difference = census_clean["total_age_18_34"] - category_sum

assert len(census_clean) == 51, "Expected 50 states and Washington, D.C."
assert census_clean["state_fips"].is_unique, "Duplicate state identifiers found."
assert census_clean.isna().sum().sum() == 0, "Missing values found."
assert (census_clean[count_columns] >= 0).all().all(), "Negative counts found."
assert (difference == 0).all(), "Living-arrangement categories do not equal the total."

print("Number of rows:", len(census_clean))
print("Missing values:", int(census_clean.isna().sum().sum()))
print("Duplicate states:", int(census_clean["state_fips"].duplicated().sum()))
print("Maximum category-total difference:", int(difference.abs().max()))

## 8. Calculate state percentages

In [ ]:
for column in category_columns:
    census_clean[f"pct_{column}"] = (
        census_clean[column] / census_clean["total_age_18_34"] * 100
    ).round(2)

display(census_clean.head())

## 9. Save the cleaned dataset

In [ ]:
clean_filename = Path("census_acs_2024_living_arrangements_clean.csv")
census_clean.to_csv(clean_filename, index=False)

print("Clean dataset shape:", census_clean.shape)
print("Saved:", clean_filename)
print("Census API collection, cleaning, and validation are complete.")

## Data source

- [U.S. Census Bureau ACS 2024 one-year API](https://api.census.gov/data/2024/acs/acs1.html)
- [ACS variable group B09021](https://api.census.gov/data/2024/acs/acs1/groups/B09021.html)